# Results downloader

Runs the **filter campaign** and the **power scan** end to end and saves every relevant figure
(and the numeric tables) into the project `results/` folder, ready to send.

```
results/
  campaign_filter/
    summary_table.txt
    comparison/   ->  overlaid g2 / R / coherence of the filters
    per_run/<run>/->  coherence, g2 and R matrices for each run
  powerscan/
    summary_table.txt , data.csv
    model/        ->  overview, intensity scaling, local slope, g2 vs power, collapse
    matrices/     ->  g2 / R vs power, coherence & g2-sweep matrices
```

Just **Run All**. Figures are rendered off-screen (Agg backend) and written to disk; nothing is
displayed inline, the cells only print the saved paths.

In [1]:
import matplotlib
matplotlib.use("Agg")   # render to files only; keeps figures open so we can save them
import matplotlib.pyplot as plt
import warnings, io, contextlib, re, csv
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive")

import numpy as np
from pathlib import Path
import sys 
import os

project_root = os.path.abspath('..')

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.hbt_core import HBTMeasurement
from src.hbt_visu import GridVisualizer
from src.hbt_powerscan import PowerScanAnalyzer

RESULTS = Path("results")
saved = []

def slug(s):
    s = s.replace('/', '-')
    return re.sub(r'[^0-9A-Za-z.\-]+', '_', s).strip('_')

def save_fig(fig, relpath, dpi=300):
    p = RESULTS / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(p, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    saved.append(p); print("  saved", p.as_posix())

def save_text(relpath, render):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        render()
    p = RESULTS / relpath
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(buf.getvalue())
    saved.append(p); print("  saved", p.as_posix())

print("Results will be written under:", RESULTS.resolve())

Results will be written under: /Users/simon.wtmn/Desktop/Quantum_SHHG/results


## 1. Filter campaign

Auto-discovers the `g2_heralded_virtual` runs (merged files), saves the summary table, the
per-run matrices and the overlaid filter comparison.

In [2]:
# CAMPAIGN_DIR = Path("data/Jun15")

# camp_files = sorted(CAMPAIGN_DIR.glob("*g2_heralded_virtual/MERGED/*_MERGED.pkl"))
# if not camp_files:
#     camp_files = sorted(CAMPAIGN_DIR.glob("*g2_heralded_virtual/*chunk0.pkl"))
# assert camp_files, f"No campaign runs found under {CAMPAIGN_DIR}"
# camp_runs = [HBTMeasurement(f) for f in camp_files]
# print(f"Campaign: {len(camp_runs)} run(s)")
# for r in camp_runs:
#     print("  -", r.short_tag())

In [3]:
# # --- summary table ---
# def _camp_table():
#     def auto(r, h):    return r.compute_g2_direct(r.get_ch(f"H{h}T"), r.get_ch(f"H{h}R"))
#     def cross(r, a, b):return r.compute_g2_direct(r.get_ch(f"H{a}T"), r.get_ch(f"H{b}T"))
#     def Rpar(r, a, b): return r.compute_R_parameter(cross(r, a, b), auto(r, a), auto(r, b))
#     cols = ["g2_33","g2_44","g2_55","g2_34","g2_35","g2_45","R_34","R_35","R_45"]
#     hdr = f"{'dataset':34s}" + "".join(f"{c:>9s}" for c in cols)
#     print(hdr); print("-"*len(hdr))
#     for r in camp_runs:
#         vals = [auto(r,3),auto(r,4),auto(r,5), cross(r,3,4),cross(r,3,5),cross(r,4,5),
#                 Rpar(r,3,4),Rpar(r,3,5),Rpar(r,4,5)]
#         print(f"{r.short_tag():34s}" + "".join(f"{v:9.3f}" for v in vals))

# save_text("campaign_filter/summary_table.txt", _camp_table)

# # --- per-run matrices ---
# for r in camp_runs:
#     sub = f"campaign_filter/per_run/{slug(r.short_tag())}"
#     print(f"RUN {r.short_tag()} ->")
#     v = GridVisualizer(r, show_details=False)
#     save_fig(v.plot_coherence(time_window_ns=5, xlim=200, integration_window_ns=10)[0], f"{sub}/coherence_matrix.png")
#     save_fig(v.plot_g2(methods=['direct','delay','heralded'], tau_min=0.3, tau_max=40, step=2.0)[0], f"{sub}/g2_matrix.png")
#     save_fig(v.plot_R(methods=['direct','delay','heralded'], tau_min=0.3, tau_max=20, step=2.0)[0], f"{sub}/R_matrix.png")

In [4]:
# # --- overlaid filter comparison ---
# print("COMPARISON ->")
# cmp = GridVisualizer(camp_runs, comparison_variable="Filter")
# save_fig(cmp.plot_coherence(time_window_ns=5, xlim=200, integration_window_ns=10)[0], "campaign_filter/comparison/coherence_matrix.png")
# save_fig(cmp.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.0)[0],        "campaign_filter/comparison/g2_delay.png")
# save_fig(cmp.plot_g2(methods=['direct'], tau_min=0.3, tau_max=25, step=1.0)[0],       "campaign_filter/comparison/g2_direct.png")
# save_fig(cmp.plot_R(methods=['delay'], tau_min=0.3, tau_max=20, step=1.0)[0],         "campaign_filter/comparison/R_delay.png")

## 2. Power scan

Saves the numeric table + CSV, the five model figures (overview, intensity scaling, local slope,
$g^{(2)}$ vs power, collapse) and the per-pair matrices vs power.

## JUN16 GaAs100 P1

In [5]:
SCAN_DIR = Path("data/Jun16/PowerScan_GaAs100_P1")
TAU_INT  = 4

scan_files = sorted(SCAN_DIR.glob("MERGED/*_MERGED.pkl"))
if not scan_files:
    scan_files = sorted(SCAN_DIR.glob("*chunk0.pkl"))
assert scan_files, f"No power-scan runs found under {SCAN_DIR}"
scan_runs = sorted([HBTMeasurement(f) for f in scan_files], key=lambda r: r.power_mw)
psa = PowerScanAnalyzer(scan_runs, harmonics=(3,4,5), tau_in_ns=TAU_INT, g2_method='delay')
print(f"Power scan: {len(scan_runs)} powers -> {[r.power_mw for r in scan_runs]}")

Power scan: 6 powers -> [10.056, 21.758, 36.906, 53.281, 81.998, 94.882]


In [6]:
# --- tables ---
save_text("powerscan_Jun16_GaAs100_P1/summary_table.txt", psa.summary_table)

def _scan_csv():
    p = RESULTS / "powerscan_Jun16_GaAs100_P1/data.csv"; p.parent.mkdir(parents=True, exist_ok=True)
    h = psa.harmonics
    with open(p, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["P_mW"] + [f"I{n}" for n in h] + [f"K{n}" for n in h]
                   + [f"g2_{n}{n}" for n in h] + [f"collapse_{n}" for n in h])
        coll = {n: psa.collapse_auto(n) for n in h}
        for i in range(len(psa.I0)):
            w.writerow([psa.I0[i]] + [psa.In[n][i] for n in h] + [psa.K[n][i] for n in h]
                       + [psa.g2_auto[n][i] for n in h] + [coll[n][i] for n in h])
    saved.append(p); print("  saved", p.as_posix())
_scan_csv()

  saved results/powerscan_Jun16_GaAs100_P1/summary_table.txt
  saved results/powerscan_Jun16_GaAs100_P1/data.csv


In [7]:
# --- model figures ---
print("MODEL ->")
save_fig(psa.plot_intensity_scaling(n_fit=1, harmonics=(3,5))[0], 
         "powerscan_Jun16_GaAs100_P1/model/intensity_scaling.png")

save_fig(psa.plot_local_slope()[0],                               
         "powerscan_Jun16_GaAs100_P1/model/local_slope.png")

save_fig(psa.plot_g2_vs_power(ylim=(0.8, 1.3), harmonics=(3,5), include_cross=True)[0],  
         "powerscan_Jun16_GaAs100_P1/model/g2_vs_power.png")

save_fig(psa.plot_g2_collapse(slope='local', harmonics=(3,5), include_cross=True)[0],                  
         "powerscan_Jun16_GaAs100_P1/model/g2_collapse.png")

# --- matrices vs power ---
print("MATRICES ->")
comp = GridVisualizer(scan_runs, comparison_variable="Pump power")
save_fig(comp.plot_power_scan_g2(tau_in_ns=TAU_INT, method='delay')[0], 
         "powerscan_Jun16_GaAs100_P1/matrices/g2_vs_power_matrix.png")

save_fig(comp.plot_power_scan_R(tau_in_ns=TAU_INT, method='delay', ylim=(0.96, 1.02))[0],  
         "powerscan_Jun16_GaAs100_P1/matrices/R_vs_power_matrix.png")

save_fig(comp.plot_coherence(time_window_ns=5, xlim=150, integration_window_ns=10)[0], 
         "powerscan_Jun16_GaAs100_P1/matrices/coherence_matrix.png")

save_fig(comp.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.5)[0],        
         "powerscan_Jun16_GaAs100_P1/matrices/g2_sweep_matrix.png")

MODEL ->
  saved results/powerscan_Jun16_GaAs100_P1/model/intensity_scaling.png
  saved results/powerscan_Jun16_GaAs100_P1/model/local_slope.png
  saved results/powerscan_Jun16_GaAs100_P1/model/g2_vs_power.png
  saved results/powerscan_Jun16_GaAs100_P1/model/g2_collapse.png
MATRICES ->
  saved results/powerscan_Jun16_GaAs100_P1/matrices/g2_vs_power_matrix.png
  saved results/powerscan_Jun16_GaAs100_P1/matrices/R_vs_power_matrix.png
  saved results/powerscan_Jun16_GaAs100_P1/matrices/coherence_matrix.png
  saved results/powerscan_Jun16_GaAs100_P1/matrices/g2_sweep_matrix.png


## JUN16 GaAs100 noP1

In [8]:
SCAN_DIR = Path("data/Jun16/PowerScan_GaAs100_noP1")
TAU_INT  = 4

scan_files = sorted(SCAN_DIR.glob("MERGED/*_MERGED.pkl"))
if not scan_files:
    scan_files = sorted(SCAN_DIR.glob("*chunk0.pkl"))
assert scan_files, f"No power-scan runs found under {SCAN_DIR}"
scan_runs = sorted([HBTMeasurement(f) for f in scan_files], key=lambda r: r.power_mw)
psa = PowerScanAnalyzer(scan_runs, harmonics=(3,4,5), tau_in_ns=TAU_INT, g2_method='delay')
print(f"Power scan: {len(scan_runs)} powers -> {[r.power_mw for r in scan_runs]}")

Power scan: 6 powers -> [22, 31.1, 39.8, 56.1, 73, 108]


In [9]:
# --- tables ---
save_text("powerscan_Jun16_GaAs100_noP1/summary_table.txt", psa.summary_table)

def _scan_csv():
    p = RESULTS / "powerscan_Jun16_GaAs100_noP1/data.csv"; p.parent.mkdir(parents=True, exist_ok=True)
    h = psa.harmonics
    with open(p, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["P_mW"] + [f"I{n}" for n in h] + [f"K{n}" for n in h]
                   + [f"g2_{n}{n}" for n in h] + [f"collapse_{n}" for n in h])
        coll = {n: psa.collapse_auto(n) for n in h}
        for i in range(len(psa.I0)):
            w.writerow([psa.I0[i]] + [psa.In[n][i] for n in h] + [psa.K[n][i] for n in h]
                       + [psa.g2_auto[n][i] for n in h] + [coll[n][i] for n in h])
    saved.append(p); print("  saved", p.as_posix())
_scan_csv()

  saved results/powerscan_Jun16_GaAs100_noP1/summary_table.txt
  saved results/powerscan_Jun16_GaAs100_noP1/data.csv


In [10]:
# --- model figures ---
print("MODEL ->")
save_fig(psa.plot_intensity_scaling(n_fit=1, harmonics=(3,5))[0], 
         "powerscan_Jun16_GaAs100_noP1/model/intensity_scaling.png")

save_fig(psa.plot_local_slope()[0],                               
         "powerscan_Jun16_GaAs100_noP1/model/local_slope.png")

save_fig(psa.plot_g2_vs_power(ylim=(0.9, 3.5), harmonics=(3,5), include_cross=True)[0],  
         "powerscan_Jun16_GaAs100_noP1/model/g2_vs_power.png")

save_fig(psa.plot_g2_collapse(slope='local', harmonics=(3,5), include_cross=True, ylim=(-0.01, 0.25))[0],                  
         "powerscan_Jun16_GaAs100_noP1/model/g2_collapse.png")

# --- matrices vs power ---
print("MATRICES ->")
comp = GridVisualizer(scan_runs, comparison_variable="Pump power")
save_fig(comp.plot_power_scan_g2(tau_in_ns=TAU_INT, method='delay')[0], 
         "powerscan_Jun16_GaAs100_noP1/matrices/g2_vs_power_matrix.png")

save_fig(comp.plot_power_scan_R(tau_in_ns=TAU_INT, method='delay')[0],  
         "powerscan_Jun16_GaAs100_noP1/matrices/R_vs_power_matrix.png")

save_fig(comp.plot_coherence(time_window_ns=5, xlim=150, integration_window_ns=10)[0], 
         "powerscan_Jun16_GaAs100_noP1/matrices/coherence_matrix.png")

save_fig(comp.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.5)[0],        
         "powerscan_Jun16_GaAs100_noP1/matrices/g2_sweep_matrix.png")

MODEL ->
  saved results/powerscan_Jun16_GaAs100_noP1/model/intensity_scaling.png
  saved results/powerscan_Jun16_GaAs100_noP1/model/local_slope.png
  saved results/powerscan_Jun16_GaAs100_noP1/model/g2_vs_power.png
  saved results/powerscan_Jun16_GaAs100_noP1/model/g2_collapse.png
MATRICES ->
  saved results/powerscan_Jun16_GaAs100_noP1/matrices/g2_vs_power_matrix.png
  saved results/powerscan_Jun16_GaAs100_noP1/matrices/R_vs_power_matrix.png
  saved results/powerscan_Jun16_GaAs100_noP1/matrices/coherence_matrix.png
  saved results/powerscan_Jun16_GaAs100_noP1/matrices/g2_sweep_matrix.png


## JUN17 GaAs100 noP1

In [11]:
SCAN_DIR = Path("data/Jun17/PowerScan_GaAs100_noP1")
TAU_INT  = 4

scan_files = sorted(SCAN_DIR.glob("MERGED/*_MERGED.pkl"))
if not scan_files:
    scan_files = sorted(SCAN_DIR.glob("*chunk0.pkl"))
assert scan_files, f"No power-scan runs found under {SCAN_DIR}"
scan_runs = sorted([HBTMeasurement(f) for f in scan_files], key=lambda r: r.power_mw)
psa = PowerScanAnalyzer(scan_runs, harmonics=(3,4,5), tau_in_ns=TAU_INT, g2_method='delay')
print(f"Power scan: {len(scan_runs)} powers -> {[r.power_mw for r in scan_runs]}")

Power scan: 7 powers -> [19.16, 22.83, 27.39, 32.68, 38.63, 54.97, 71.95]


In [12]:
# --- tables ---
save_text("powerscan_Jun17_GaAs100_noP1/summary_table.txt", psa.summary_table)

def _scan_csv():
    p = RESULTS / "powerscan_Jun17_GaAs100_noP1/data.csv"; p.parent.mkdir(parents=True, exist_ok=True)
    h = psa.harmonics
    with open(p, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["P_mW"] + [f"I{n}" for n in h] + [f"K{n}" for n in h]
                   + [f"g2_{n}{n}" for n in h] + [f"collapse_{n}" for n in h])
        coll = {n: psa.collapse_auto(n) for n in h}
        for i in range(len(psa.I0)):
            w.writerow([psa.I0[i]] + [psa.In[n][i] for n in h] + [psa.K[n][i] for n in h]
                       + [psa.g2_auto[n][i] for n in h] + [coll[n][i] for n in h])
    saved.append(p); print("  saved", p.as_posix())
_scan_csv()

  saved results/powerscan_Jun17_GaAs100_noP1/summary_table.txt
  saved results/powerscan_Jun17_GaAs100_noP1/data.csv


In [13]:
# --- model figures ---
print("MODEL ->")
save_fig(psa.plot_intensity_scaling(n_fit=1, harmonics=(3,5))[0], 
         "powerscan_Jun17_GaAs100_noP1/model/intensity_scaling.png")

save_fig(psa.plot_local_slope()[0],                               
         "powerscan_Jun17_GaAs100_noP1/model/local_slope.png")

save_fig(psa.plot_g2_vs_power(ylim=(0.9, 3.5), harmonics=(3,5), include_cross=True)[0],  
         "powerscan_Jun17_GaAs100_noP1/model/g2_vs_power.png")

save_fig(psa.plot_g2_collapse(slope='local', harmonics=(3,5), include_cross=True, ylim=(-0.01, 0.25))[0],                  
         "powerscan_Jun17_GaAs100_noP1/model/g2_collapse.png")

# --- matrices vs power ---
print("MATRICES ->")
comp = GridVisualizer(scan_runs, comparison_variable="Pump power")
save_fig(comp.plot_power_scan_g2(tau_in_ns=TAU_INT, method='delay')[0], 
         "powerscan_Jun17_GaAs100_noP1/matrices/g2_vs_power_matrix.png")

save_fig(comp.plot_power_scan_R(tau_in_ns=TAU_INT, method='delay')[0],  
         "powerscan_Jun17_GaAs100_noP1/matrices/R_vs_power_matrix.png")

save_fig(comp.plot_coherence(time_window_ns=5, xlim=150, integration_window_ns=10)[0], 
         "powerscan_Jun17_GaAs100_noP1/matrices/coherence_matrix.png")

save_fig(comp.plot_g2(methods=['delay'], tau_min=0.3, tau_max=25, step=1.5)[0],        
         "powerscan_Jun17_GaAs100_noP1/matrices/g2_sweep_matrix.png")

MODEL ->
  saved results/powerscan_Jun17_GaAs100_noP1/model/intensity_scaling.png
  saved results/powerscan_Jun17_GaAs100_noP1/model/local_slope.png
  saved results/powerscan_Jun17_GaAs100_noP1/model/g2_vs_power.png
  saved results/powerscan_Jun17_GaAs100_noP1/model/g2_collapse.png
MATRICES ->
  saved results/powerscan_Jun17_GaAs100_noP1/matrices/g2_vs_power_matrix.png
  saved results/powerscan_Jun17_GaAs100_noP1/matrices/R_vs_power_matrix.png
  saved results/powerscan_Jun17_GaAs100_noP1/matrices/coherence_matrix.png
  saved results/powerscan_Jun17_GaAs100_noP1/matrices/g2_sweep_matrix.png
